cvxpy 是 Python 的凸优化（convex optimization）建模库，用来求解各种优化问题，尤其是线性规划（LP）、二次规划（QP）、凸优化等。

In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
import cvxpy as cp 

from quantmod.markets import getData
import plotly.graph_objects as go

import cufflinks as cf
cf.go_offline()  # 启用离线模式

In [2]:
df = getData(['BTC-USD','ETH-USD','BNB-USD','SOL-USD','DOGE-USD'], period = '2y' )
df = df['Close']
df

Ticker,BNB-USD,BTC-USD,DOGE-USD,ETH-USD,SOL-USD
Date,,,,,
2023-08-13,240.033340,29282.914062,0.074783,1839.280151,24.242886
2023-08-14,240.339249,29408.443359,0.074731,1844.185791,25.142822
2023-08-15,236.679581,29170.347656,0.070863,1826.932739,23.896585
2023-08-16,231.868744,28701.779297,0.067183,1805.659058,22.811975
2023-08-17,218.535843,26664.550781,0.061185,1684.933472,21.709974
...,...,...,...,...,...
2025-08-08,793.439575,116688.726562,0.230214,4009.848877,176.763290
2025-08-09,799.086731,116500.359375,0.240325,4263.599121,180.182343
2025-08-10,805.873474,119306.757812,0.234099,4254.217773,182.903015


In [3]:
df.isnull().sum()

Ticker
BNB-USD     0
BTC-USD     0
DOGE-USD    0
ETH-USD     0
SOL-USD     0
dtype: int64

In [4]:
avg = df['BTC-USD'].mean()
df['BTC-USD'] = df['BTC-USD'].fillna(avg)
df = pd.DataFrame(df)
df

Ticker,BNB-USD,BTC-USD,DOGE-USD,ETH-USD,SOL-USD
Date,,,,,
2023-08-13,240.033340,29282.914062,0.074783,1839.280151,24.242886
2023-08-14,240.339249,29408.443359,0.074731,1844.185791,25.142822
2023-08-15,236.679581,29170.347656,0.070863,1826.932739,23.896585
2023-08-16,231.868744,28701.779297,0.067183,1805.659058,22.811975
2023-08-17,218.535843,26664.550781,0.061185,1684.933472,21.709974
...,...,...,...,...,...
2025-08-08,793.439575,116688.726562,0.230214,4009.848877,176.763290
2025-08-09,799.086731,116500.359375,0.240325,4263.599121,180.182343
2025-08-10,805.873474,119306.757812,0.234099,4254.217773,182.903015


In [5]:
df_normalized = df / df.iloc[0]

fig = go.Figure()

for col in df_normalized.columns:
    fig.add_trace(go.Scatter(
            x = df_normalized.index,
            y = df_normalized[col],
            name = col,
            mode = 'lines', #设置绘图模式
            line = go.scatter.Line() #设置线条样式
        ))

# 设置布局
fig.update_layout(
    title = 'Cryptos normalized Price Trends',
    xaxis_title = 'Date',
    yaxis_title = 'Normalized Price',
    showlegend = True
)

# 显示图形
fig.show()

# Black-Litterman Model

#### Step 1：计算市场隐含收益率

In [6]:
prices = df.copy().sort_index()
tickers = list(prices.columns)

# 你可以设置无风险利率（年化）。加密组合一般用USD无风险，比如3%：
Rf = 0.03

# 年化天数：加密市场全年交易，常取365
ANNUALIZE = 365

rets = np.log(prices / prices.shift(1)).dropna()
cov_matrix = (rets.cov() * ANNUALIZE).values
cov_matrix

array([[0.26512486, 0.15536154, 0.26777092, 0.21462139, 0.24744145],
       [0.15536154, 0.23275027, 0.3400859 , 0.24908625, 0.30210827],
       [0.26777092, 0.3400859 , 0.85897643, 0.44450804, 0.52471332],
       [0.21462139, 0.24908625, 0.44450804, 0.43625885, 0.39282816],
       [0.24744145, 0.30210827, 0.52471332, 0.39282816, 0.77550635]])

In [7]:
from pycoingecko import CoinGeckoAPI
cg = CoinGeckoAPI()
market_caps = cg.get_price(ids=['bitcoin','solana','ethereum','binancecoin','dogecoin'], vs_currencies='usd', include_market_cap='true')
market_caps

{'binancecoin': {'usd': 852.22, 'usd_market_cap': 118731360467.07812},
 'bitcoin': {'usd': 120549, 'usd_market_cap': 2399129129019.636},
 'dogecoin': {'usd': 0.246433, 'usd_market_cap': 37074348524.04396},
 'ethereum': {'usd': 4684.81, 'usd_market_cap': 565489869577.1882},
 'solana': {'usd': 201.01, 'usd_market_cap': 108399419755.89825}}

In [8]:
# 1) 建立 id -> 价格表列名 的映射（按你 df 的列来写）
id2col = {
    'bitcoin':      'BTC-USD',
    'ethereum':     'ETH-USD',
    'binancecoin':  'BNB-USD',
    'solana':       'SOL-USD',
    'dogecoin':     'DOGE-USD',
}

# 2) 从字典里提取市值，转成 Series，并把索引改成你价格表的列名
s_caps = pd.Series({
    id2col[k]: v.get('usd_market_cap', 0.0) for k, v in market_caps.items() if k in id2col}).astype(float)

# 3) 做成 DataFrame
df_caps = s_caps.rename('MarketCap').to_frame()
df_caps['Weight'] = df_caps['MarketCap'] / df_caps['MarketCap'].sum()
display(df_caps)

,MarketCap,Weight
BNB-USD,1.187314e+11,0.036772
BTC-USD,2.399129e+12,0.743035
DOGE-USD,3.707435e+10,0.011482
ETH-USD,5.654899e+11,0.175138
SOL-USD,1.083994e+11,0.033572


In [9]:
# 对齐到 df.columns 的顺序，缺的用 0 填，再归一化
w_m_series = df_caps['Weight'].reindex(prices.columns).fillna(0.0)
w_m = (w_m_series / w_m_series.sum()).values  # numpy 向量
print("w_m aligned:", w_m)

w_m aligned: [0.03677232 0.74303494 0.01148231 0.17513802 0.03357241]


In [10]:
# 2) 用 w_m 构造市场组合的历史收益，估计 E[Rm] 与 Var(Rm)
port_daily = rets.values @ w_m
ERm = port_daily.mean() * ANNUALIZE
Var_m = port_daily.var() * ANNUALIZE
sigma_m = Var_m**0.5

# 3) 风险厌恶 λ（CAPM 常用推法）
lam = (ERm - Rf) / Var_m
print(f"E[Rm]={ERm:.4f}, sigma_m={sigma_m:.4f}, lambda={lam:.4f}")

E[Rm]=0.6731, sigma_m=0.4975, lambda=2.5985


In [11]:
# 4) 隐含超额收益 π = λ Σ w_m
pi = lam * (cov_matrix @ w_m)                     
pi = pd.Series(pi, index = prices.columns, name='pi')                                 
display(pi)

Ticker
BNB-USD     0.452558
BTC-USD     0.614104
DOGE-USD    0.955928
ETH-USD     0.747521
SOL-USD     0.869044
Name: pi, dtype: float64

#### Step 2: 选择信号

In [12]:
# 信号：180天动量 
lookback = 180
rets = np.log(prices/prices.shift(1))
mom = np.log(prices/prices.shift(lookback)).replace([np.inf,-np.inf], np.nan)

In [13]:
# 90天前瞻收益（校准用）
h = 90
fwd = np.log(prices.shift(-h)/prices).replace([np.inf,-np.inf], np.nan)

#### 以下代码的目的，是整理训练样本，把：

- 特征 (X) → 每个币在某个时间点的动量信号（mom）
- 标签 (y) → 对应的未来 90 天收益率（fwd）

拼接成一个适合机器学习训练的二维数组，并进行标准化（z-score）。

In [14]:
X, y = [], []

for c in prices.columns:
    df_c = pd.concat(
        [mom[c].rename("mom"), fwd[c].rename("fwd")],
        axis=1
    ).dropna()

    print(c, len(df_c))

    if len(df_c) == 0:
        continue

    X.append(df_c["mom"].to_numpy())
    y.append(df_c["fwd"].to_numpy())

X = np.concatenate(X)
y = np.concatenate(y)

mask = np.isfinite(X) & np.isfinite(y)
X, y = X[mask], y[mask]

Xz = (X - np.nanmean(X)) / np.nanstd(X)

BNB-USD 461
BTC-USD 461
DOGE-USD 461
ETH-USD 461
SOL-USD 461


In [15]:
df_c

,mom,fwd
Date,,
2024-02-09,1.484082,0.357251
2024-02-10,1.467981,0.292538
2024-02-11,1.503711,0.301453
2024-02-12,1.591149,0.248046
2024-02-13,1.645886,0.268693
...,...,...
2025-05-10,-0.225907,-0.004845
2025-05-11,-0.204235,0.040525
2025-05-12,-0.211017,0.048477


In [16]:
#生成设计矩阵
Xmat = np.vstack([np.ones_like(Xz), Xz]).T

#最小二乘解 
coef = np.linalg.lstsq(Xmat, y, rcond=None)[0]

#拆出系数
a, b = coef

#残差反映模型没解释掉的部分（噪声、非线性、特征不足等）
resid = y - (a + b * Xz)

#残差方差 / MSE
mse = np.var(resid, ddof=2)  # 两个参数 a、b 已经被估计，扣掉 2 个自由度

print(f"Coeif: a={a:.4f}, b={b:.4f}, MSE(90d)={mse:.6f}")

Coeif: a=0.0478, b=-0.1735, MSE(90d)=0.117024


#### 用“当前信号”生成观点收益（年化）

In [17]:
# 计算当前动量的 z 分数 -> 为了消除每个币种间的量纲问题，转为可比较的标准尺度，方便后续预测标准化后的信号强弱
curr_sig = ((mom.iloc[-1] - mom.stack().mean()) / mom.stack().std()).reindex(prices.columns)

# 预测 -> 把今天的动能强弱映射到未来90天的预测收益 
pred_90d = a + b * curr_sig 

# 年化收益 -> Black-Litterman 模型里的观点收益通常要用年化值，这样和市场风险溢价等量纲一致
anul_pred_90d = pred_90d * (ANNUALIZE / h)

display(curr_sig.to_frame("Z Score").style.format("{:.2f}"))
display(anul_pred_90d.to_frame("Annual Return").style.format("{:.2f}"))

,Z Score
Ticker,
BNB-USD,-0.09
BTC-USD,-0.14
DOGE-USD,-0.70
ETH-USD,0.52
SOL-USD,-0.51


,Annual Return
Ticker,
BNB-USD,0.26
BTC-USD,0.29
DOGE-USD,0.68
ETH-USD,-0.18
SOL-USD,0.55


#### 现实业界怎么选观点 通常会：

- 聚焦在预测值排名前 20% 和后 20% 的资产
- 用高预测 − 低预测 做相对观点
- 如果有特别高的预测值（比如你这里 SOL 和 DOGE），会加绝对观点

In [18]:
views = [
    # 相对观点：SOL 高于 BNB 约 0.51
    {"type": "rel", "long": "SOL-USD", "short": "BNB-USD", "q": 0.73 - 0.25},

    # 相对观点：BTC 高于 ETH 约 0.32
    {"type": "rel", "long": "BTC-USD", "short": "ETH-USD", "q": 0.34 - 0.02},

    # 绝对观点：DOGE 的收益为 0.72
    {"type": "abs", "asset": "DOGE-USD", "q": 0.72},

    # 绝对观点：SOL 的收益为 0.73
    {"type": "abs", "asset": "SOL-USD", "q": 0.73}
]

In [19]:
# --- 用 rets 计算每个观点组合的年化波动率 & 置信度 conf ---
conf_list, sigma_list = [], []
for v in views:
    if v["type"] == "rel":
        # 组合 = long - short
        combo = rets[v["long"]] - rets[v["short"]]
    elif v["type"] == "abs":
        combo = rets[v["asset"]]
    else:
        raise ValueError("view type 必须是 'abs' 或 'rel'")

    # 清洗并年化
    combo = combo.dropna()
    sigma_annual = combo.std() * np.sqrt(ANNUALIZE)   # 与 cov_matrix 的年化口径一致
    conf = float(v["q"]) / float(sigma_annual) if sigma_annual > 0 else np.nan

    sigma_list.append(sigma_annual)
    conf_list.append(conf)

df_conf = pd.DataFrame(views)
df_conf["sigma_annual"] = sigma_list
df_conf["conf"] = conf_list
df_conf

,type,long,short,q,asset,sigma_annual,conf
0,rel,SOL-USD,BNB-USD,0.48,NaN,0.738748,0.649748
1,rel,BTC-USD,ETH-USD,0.32,NaN,0.413324,0.774211
2,abs,NaN,NaN,0.72,DOGE-USD,0.926810,0.776858
3,abs,NaN,NaN,0.73,SOL-USD,0.880628,0.828954


In [20]:
# --- 资产顺序与映射 ---
tickers = list(prices.columns)
idx = {t:i for i,t in enumerate(tickers)}
N = len(tickers)

def one_hot(asset):
    v = np.zeros(N); v[idx[asset]] = 1.0
    return v

print(tickers)
print(idx)
print(one_hot(tickers[0]))   # 第一只资产的单位向量

['BNB-USD', 'BTC-USD', 'DOGE-USD', 'ETH-USD', 'SOL-USD']
{'BNB-USD': 0, 'BTC-USD': 1, 'DOGE-USD': 2, 'ETH-USD': 3, 'SOL-USD': 4}
[1. 0. 0. 0. 0.]


In [21]:
# --- 从 views观点列表中重建 P, q ---
P_rows, q_list = [], []
for v in views:
    if v["type"] == "rel":
        P_rows.append(one_hot(v["long"]) - one_hot(v["short"]))
        q_list.append(float(v["q"]))
    elif v["type"] == "abs":
        P_rows.append(one_hot(v["asset"]))
        q_list.append(float(v["q"]))
    else:
        raise ValueError("view type must be 'abs' 或 'rel'")

P = np.vstack(P_rows) # K x N
q = np.array(q_list, dtype=float).reshape(-1, 1)  # K x 1
K = P.shape[0]

print("P shape:", P.shape, "q shape:", q.shape, "K:", K)
print(pd.DataFrame(P, columns=tickers).assign(q=q.ravel()))

P shape: (4, 5) q shape: (4, 1) K: 4
   BNB-USD  BTC-USD  DOGE-USD  ETH-USD  SOL-USD     q
0     -1.0      0.0       0.0      0.0      1.0  0.48
1      0.0      1.0       0.0     -1.0      0.0  0.32
2      0.0      0.0       1.0      0.0      0.0  0.72
3      0.0      0.0       0.0      0.0      1.0  0.73


In [22]:
# --- 协方差 Σ (N * N)---
Sigma = np.array(cov_matrix, dtype=float)
print("Sigma shape:", Sigma.shape)

Sigma shape: (5, 5)


In [23]:
# --- 线性缩放的 Ω ---
conf = df_conf["conf"].astype(float).values      # 长度 K，与 views 对齐
# view_var = np.einsum('ki,ij,kj->k', P, Sigma, P) # diag(P Σ P^T) 的高效写法
view_var = np.diag(P @ Sigma @ P.T)
Omega_diag = view_var / conf                     # 线性：除以 conf
Omega = np.diag(Omega_diag + 1e-12)              # 数值稳健性保护

pd.DataFrame({"view_var":view_var, "conf":conf, "Omega_diag":Omega_diag})

,view_var,conf,Omega_diag
0,0.545748,0.649748,0.839938
1,0.170837,0.774211,0.220659
2,0.858976,0.776858,1.105705
3,0.775506,0.828954,0.935525


In [24]:
# --- 先验缩放参数 tau（常用 0.025~0.05，可按需调整 /之后做敏感性分析）---
tau = 0.05

In [25]:
# --- 计算加权后更新的预期收益率（μ_BL） ---
Sigma_tau_inv = np.linalg.inv(tau * Sigma)
Omega_inv      = np.linalg.inv(Omega)

A = Sigma_tau_inv + P.T @ Omega_inv @ P
b = Sigma_tau_inv @ pi.values.reshape(-1,1) + P.T @ Omega_inv @ q

mu_bl = np.linalg.solve(A, b) # 用 solve 比直接求逆更稳健
mu_bl = pd.Series(mu_bl.ravel(), index=tickers, name="mu_BL")

print("A shape:", A.shape, "b shape:", b.shape, "mu_bl shape:", mu_bl.values.shape)
display(pd.DataFrame({"pi (prior)": pi, "mu_BL": mu_bl}).style.format("{:.4f}"))

A shape: (5, 5) b shape: (5, 1) mu_bl shape: (5,)


,pi (prior),mu_BL
BNB-USD,0.4526,0.4424
BTC-USD,0.6141,0.6078
DOGE-USD,0.9559,0.9349
ETH-USD,0.7475,0.7229
SOL-USD,0.8690,0.8521


In [35]:
def optimize_max_sharpe_bl(mu_bl, Sigma, Rf=0.0, wmax=1.0):
    mu = mu_bl.values.reshape(-1, 1)
    n  = mu.shape[0]
    w  = cp.Variable(n)

    port_excess = (mu - Rf).T @ w
    port_var    = cp.quad_form(w, Sigma)          # 二次
    constraints = [
        cp.sum(w) == 1,
        w >= 0,
        w <= wmax,
        port_var <= 1.0
    ]
    prob = cp.Problem(cp.Maximize(port_excess), constraints)
    prob.solve(solver=cp.SCS)   # 或 ECOS/OSQP
    return pd.Series(np.array(w.value).ravel(), index=mu_bl.index, name="w_max_sharpe")


In [39]:
# 假设 mu_bl, Sigma 是你前面 Black-Litterman 得到的后验均值和协方差矩阵
w_max_sharpe = optimize_max_sharpe_bl(mu_bl, Sigma, Rf=0.03, wmax=0.35)

print(w_max_sharpe)  # 打印权重结果

BNB-USD    -5.408688e-07
BTC-USD    -2.947296e-07
DOGE-USD    3.500008e-01
ETH-USD     2.999996e-01
SOL-USD     3.500004e-01
Name: w_max_sharpe, dtype: float64


In [37]:
w = w_max_sharpe.copy().astype(float)
w[w < 0] = 0.0
w = w / w.sum()
print(w)

BNB-USD     0.000000
BTC-USD     0.000000
DOGE-USD    0.399995
ETH-USD     0.200014
SOL-USD     0.399991
Name: w_max_sharpe, dtype: float64
